In [8]:
import pandas as pd
import numpy as np
from pybaseball import statcast, batting_stats, playerid_reverse_lookup
from tqdm.auto import tqdm
import os

# Initialize tqdm for pandas
tqdm.pandas()

# 1. Directory Setup
current_dir = os.getcwd()
base_dir = os.path.dirname(current_dir) 
output_dir = os.path.join(base_dir, 'data', 'processed')
os.makedirs(output_dir, exist_ok=True)

# 2. Load 2024 Statcast Data
# (Assuming 'raw_data' was downloaded previously)

if not raw_data.empty:
    # Normalize column names to lowercase
    raw_data.columns = raw_data.columns.str.lower()

    # 3. Aggregate Physical & Behavioral Metrics (Statcast Exclusive)
    def aggregate_physical_stats(group):
        # Calculate Plate Appearances (PA) for initial filtering
        pa = group.groupby(['game_date', 'at_bat_number']).size().shape[0]
        if pa < 100: return None
        
        # [Classification Metrics] Bat Speed & Contact %
        bat_speed = group['bat_speed'].fillna(group['launch_speed'] * 0.75).mean()
        
        swings = group[group['description'].str.contains('swinging_strike|foul|hit_into_play', na=False)]
        misses = group[group['description'].str.contains('swinging_strike', na=False)]
        contact_pct = (len(swings) - len(misses)) / len(swings) if len(swings) > 0 else 0
        
        # [Rank/Quality Metrics] Behavioral data
        chase_opps = group[group['zone'] > 10]
        chase_pct = len(chase_opps[chase_opps['description'].str.contains('swing|foul|hit_into_play', na=False)]) / len(chase_opps) if len(chase_opps) > 0 else 0
        
        avg_ev = group['launch_speed'].mean()
        la_valid = group['launch_angle'].dropna()
        sweet_spot_pct = len(la_valid[(la_valid >= 8) & (la_valid <= 32)]) / len(la_valid) if len(la_valid) > 0 else 0
        
        xwoba = group['estimated_woba_using_speedangle'].mean()

        return pd.Series({
            'avg_bat_speed': bat_speed, 
            'contact_pct': contact_pct,
            'chase_pct': chase_pct, 
            'avg_ev': avg_ev, 
            'sweet_spot_pct': sweet_spot_pct, 
            'xwoba': xwoba
        })

    print("Aggregating physical metrics from Statcast (This may take a while)...")
    
    physical_df = raw_data.groupby('batter').progress_apply(aggregate_physical_stats, include_groups=False).dropna().reset_index()

    # 4. Player ID & Name Mapping
    print("🔗 Mapping player IDs...")
    unique_ids = physical_df['batter'].unique().tolist()
    lookup_df = playerid_reverse_lookup(unique_ids, key_type='mlbam')
    lookup_df['full_name'] = lookup_df['name_first'].str.capitalize() + " " + lookup_df['name_last'].str.capitalize()
    
    physical_master = physical_df.merge(lookup_df[['key_mlbam', 'full_name']], left_on='batter', right_on='key_mlbam', how='left')

    # ---------------------------------------------------------
    # 5. Collect Official Performance Stats (Hybrid Strategy)
    # ---------------------------------------------------------
    print("🌐 Fetching official season stats from pybaseball...")

    # Set 'qual' parameter to 150 (or 200) to get more than just qualified hitters
    official_stats = batting_stats(2024, qual=150).copy() 

    print(f"✅ Retrieved {len(official_stats)} players from pybaseball (qual=150)")

    # [XBH calculation and column renaming logic - same as before]
    if 'XBH' not in official_stats.columns:
        official_stats['XBH'] = official_stats['2B'] + official_stats['3B'] + official_stats['HR']
    
    required_cols = ['Name', 'PA', 'AVG', 'OBP', 'SLG', 'OPS', 'wOBA', 'ISO', 'K%', 'BB%', 'XBH']
    official_stats = official_stats[required_cols].copy()
    official_stats.columns = ['full_name', 'pa', 'avg', 'obp', 'slg', 'ops', 'woba_official', 'iso', 'k_pct', 'bb_pct', 'xbh']
    # ---------------------------------------------------------
    # 6. Final Integration
    # ---------------------------------------------------------
    print("📊 Merging physical inputs with performance outputs...")
    final_master = pd.merge(physical_master, official_stats, on='full_name', how='inner')
    final_master = final_master[final_master['pa'] >= 200].reset_index(drop=True)
    final_master = final_master.drop(columns=['key_mlbam', 'batter'])

    # 7. Export Final Result
    output_path = os.path.join(output_dir, 'hitter_master_data.csv')
    final_master.to_csv(output_path, index=False)
    
    print("\n" + "="*50)
    print(f"✅ Data Collection Completed!")
    print(f"📊 Final Sample: {len(final_master)} hitters")
    print("="*50)
    display(final_master.head())

⏳ Aggregating physical metrics from Statcast (This may take a while)...


  0%|          | 0/1223 [00:00<?, ?it/s]

🔗 Mapping player IDs...
🌐 Fetching official season stats from pybaseball...
✅ Retrieved 410 players from pybaseball (qual=150)
📊 Merging physical inputs with performance outputs...

✅ Data Collection Completed!
📊 Final Sample: 290 hitters


,avg_bat_speed,contact_pct,chase_pct,avg_ev,sweet_spot_pct,xwoba,full_name,pa,avg,obp,slg,ops,woba_official,iso,k_pct,bb_pct,xbh
0,69.482651,0.758095,0.325758,82.789655,0.310345,0.318220,David Peralta,260,0.267,0.335,0.415,0.750,0.329,0.148,0.208,0.085,19
1,65.881020,0.829294,0.293456,80.283613,0.289510,0.308825,Charlie Blackmon,499,0.256,0.329,0.412,0.741,0.323,0.156,0.172,0.086,41
2,65.386933,0.787022,0.297907,81.912442,0.320276,0.312522,Donovan Solano,309,0.286,0.343,0.417,0.760,0.333,0.131,0.210,0.071,21
3,63.614966,0.847866,0.252560,81.431759,0.327037,0.333037,Justin Turner,539,0.259,0.354,0.383,0.737,0.327,0.124,0.176,0.109,35
4,70.015839,0.788931,0.234009,83.721932,0.277344,0.332602,Carlos Santana,594,0.238,0.328,0.420,0.749,0.326,0.182,0.167,0.109,49
